# 10 · Обучение XLS-R
Начните с Frozen Encoder. Длительное обучение запускается только вручную.

In [ ]:
import subprocess
import sys
from pathlib import Path

repo=Path('/content/audio-deepfake-detector')
if not (repo/'.git').exists(): subprocess.run(['git','clone','https://github.com/Unvand/audio-deepfake-detector.git',str(repo)],check=True)
else: subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(repo/'requirements-colab-train.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44,<5'],check=True)
sys.path.insert(0,str(repo))
commit=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
from google.colab import drive

drive.mount('/content/drive')
base=Path('/content/drive/MyDrive/VKR_RuASD')
prepared=base/'prepared_v1'
print('Git commit:',commit)

In [ ]:
import torch

from audio_deepfake_detector.src.xlsr_training import XLSRConfig, audit_manifest, train

frame,summary=audit_manifest(prepared/'ruasd_segments_manifest.csv')
display(summary)
print('GPU:',torch.cuda.is_available())
print('XLS-R 300m может не поместиться на T4 с большим batch; начните с batch=1–2 и accumulation.')

In [ ]:
import ipywidgets as widgets

mode=widgets.Dropdown(options=['frozen','partial'],value='frozen',description='Режим')
batch=widgets.IntSlider(value=2,min=1,max=8,description='Batch')
epochs=widgets.IntSlider(value=5,min=1,max=50,description='Эпохи')
augment=widgets.FloatSlider(value=0,min=0,max=.5,step=.05,description='Аугментации')
resume=widgets.Checkbox(value=False,description='Resume')
run_id=widgets.Text(value='xlsr_baseline_v1',description='run_id')
display(mode,batch,epochs,augment,resume,run_id)

In [ ]:
# Явный smoke test: несколько train/validation фрагментов и один шаг Frozen Encoder
RUN_SMOKE_TEST=False
if RUN_SMOKE_TEST:
 from audio_deepfake_detector.src.xlsr_training import WaveformDataset, XLSRDetector
 small=frame.groupby(['split','label'],group_keys=False).head(1)
 cfg=XLSRConfig(batch_size=1,epochs=1)
 model=XLSRDetector.from_pretrained(cfg)
 dataset=WaveformDataset(small,prepared,'train',cfg)
 wave,mask,label,_=dataset[0]
 before=model.head[1].weight.detach().clone()
 opt=torch.optim.AdamW(model.head.parameters(),lr=cfg.head_lr)
 loss=torch.nn.functional.binary_cross_entropy_with_logits(model(wave[None],mask[None]),label[None])
 loss.backward();opt.step()
 assert not torch.equal(before,model.head[1].weight)
 assert all(p.grad is None for p in model.encoder.parameters())
 print('Smoke loss:',float(loss),'память CUDA (МиБ):',torch.cuda.max_memory_allocated()/2**20 if torch.cuda.is_available() else 'CPU')

In [ ]:
RUN_TRAINING=False
if RUN_TRAINING:
 cfg=XLSRConfig(mode=mode.value,batch_size=batch.value,epochs=epochs.value,augmentation_probability=augment.value)
 out=base/'training_xlsr_v1/runs'/run_id.value.strip()
 history=train(prepared,frame,out,cfg,resume=resume.value,git_commit=commit)
 print('Сохранено:',out,'эпох:',len(history))